In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error
import scipy.sparse as sp
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('/content/Reviews.csv', usecols=['UserId','ProductId','Score'], on_bad_lines='skip', encoding='latin-1', engine='python')
df.columns = ['user_id','product_id','rating']
print("Rows:", df.shape[0])
df.head()


Rows: 545926


,user_id,product_id,rating
0,B001E4KFG0,A3SGXH7AUHU8GW,5
1,B00813GRG4,A1D87F6ZCVE5NK,1
2,B000LQOCH0,ABXLMWJIXXAIN,4
3,B000UA0QIQ,A395BORC6FGVXV,2
4,B006K2ZZ7K,A1UQRSCLF8GW1T,5


In [ ]:
# keep users and products with some minimum interactions (optional)
min_user_ratings = 5
min_product_ratings = 5

# aggregate if duplicate user-product pairs exist (take mean rating)
df = df.groupby(['user_id','product_id'], as_index=False).rating.mean()

# filter
user_counts = df['user_id'].value_counts()
product_counts = df['product_id'].value_counts()

valid_users = user_counts[user_counts >= min_user_ratings].index
valid_products = product_counts[product_counts >= min_product_ratings].index

df = df[df['user_id'].isin(valid_users) & df['product_id'].isin(valid_products)].reset_index(drop=True)
print("Filtered rows:", df.shape[0])


Filtered rows: 198582


In [ ]:
# create id -> index mappings
user_ids = df['user_id'].unique()
prod_ids = df['product_id'].unique()
user_to_idx = {u:i for i,u in enumerate(user_ids)}
prod_to_idx = {p:i for i,p in enumerate(prod_ids)}
idx_to_prod = {i:p for p,i in prod_to_idx.items()}

rows = df['user_id'].map(user_to_idx)
cols = df['product_id'].map(prod_to_idx)
values = df['rating'].values

n_users = len(user_ids)
n_products = len(prod_ids)
print("Users:", n_users, "Products:", n_products)

# build sparse matrix (users x products)
R = sp.csr_matrix((values, (rows, cols)), shape=(n_users, n_products))


Users: 16757 Products: 21818


In [ ]:
# Compute item vectors (product x user)
item_user_mat = R.T.tocsr()  # shape: n_products x n_users

# Convert to dense? Avoid if too large. Compute pairwise cosine similarity in sparse form.
# We'll compute cosine similarity using sklearn on dense rows if product count is small.
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# For memory safety, convert to dense only if n_products small. Otherwise use a chunked approach.
if n_products <= 5000:
    item_dense = item_user_mat.toarray()
    item_sim = cosine_similarity(item_dense)   # item-item similarity matrix
else:
    # compute similarities in chunks to avoid memory blowup (simple approximate approach)
    item_dense = item_user_mat.toarray()
    item_sim = cosine_similarity(item_dense)   # try anyway but be careful with memory
print("Item similarity matrix shape:", item_sim.shape)


Item similarity matrix shape: (21818, 21818)


In [ ]:
def get_user_top_n_recs(user_id, top_n=10):
    """
    Score items for a user using item-based CF:
    predicted_rating_i = sum_over_j(sim(i,j) * r_u_j) / sum_over_j(|sim(i,j)|)
    where r_u_j are items user has rated.
    """
    if user_id not in user_to_idx:
        return []
    uidx = user_to_idx[user_id]
    user_ratings = R[uidx].toarray().flatten()  # length = n_products
    rated_idx = np.where(user_ratings > 0)[0]
    if len(rated_idx) == 0:
        return []
    scores = np.zeros(n_products, dtype=float)
    sim_sums = np.zeros(n_products, dtype=float)
    for j in rated_idx:
        sim_j = item_sim[:, j]           # similarity of all items with item j
        scores += sim_j * user_ratings[j]
        sim_sums += np.abs(sim_j)
    # avoid dividing by zero
    with np.errstate(divide='ignore', invalid='ignore'):
        preds = np.divide(scores, sim_sums)
    preds[rated_idx] = -np.inf  # don't recommend already-rated items
    top_idx = np.argsort(preds)[-top_n:][::-1]
    return [idx_to_prod[i] for i in top_idx if preds[i] != -np.inf][:top_n]

def get_similar_products(product_id, top_n=10):
    if product_id not in prod_to_idx:
        return []
    pidx = prod_to_idx[product_id]
    sims = item_sim[pidx]
    top_idx = np.argsort(sims)[-top_n-1:][::-1]  # includes itself
    top_idx = [i for i in top_idx if i != pidx][:top_n]
    return [idx_to_prod[i] for i in top_idx]


In [ ]:
# Pick a random user and product
sample_user = user_ids[0]
sample_prod = prod_ids[0]
print("Sample user:", sample_user)
print("Top recommendations for user:", get_user_top_n_recs(sample_user, top_n=10))
print("Products similar to sample product:", get_similar_products(sample_prod, top_n=10))


Sample user: 0006641040
Top recommendations for user: ['A1MV4JYOZEDGNX', 'A3RD82GSN0C6B2', 'A3DVYC3WFZ07ET', 'AMAM0LTSSX7SB', 'A1YLOZQKBX3J1S', 'A3PRVY5OU76U4V', 'A22S0RZ11PP3S3', 'A27E44IWS3B7VD', 'AVZ1SYMD26S4R', 'A398OWPQ0U18R0']
Products similar to sample product: ['A37E6RW5BUX4U0', 'A1OGXR0166Q0XJ', 'A190EAFTGCQNED', 'A1RLA4JD522O0S', 'A3CWEK9HY2VPDL', 'A1VKAY0XWLA2CY', 'A2ZJMLZ1IA2YA9', 'AJKWF4W7QD4NS', 'ALEUQQE118NEW', 'A3464G00K8ZYD1']


In [ ]:
# Create train/test split at the rating-row level
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# rebuild train matrix
rows_train = train_df['user_id'].map(user_to_idx)
cols_train = train_df['product_id'].map(prod_to_idx)
vals_train = train_df['rating'].values
R_train = sp.csr_matrix((vals_train, (rows_train, cols_train)), shape=(n_users, n_products))

# recompute item_sim using R_train
item_user_mat_train = R_train.T.tocsr()
item_dense_train = item_user_mat_train.toarray()
item_sim_train = cosine_similarity(item_dense_train)

# predict helper using item_sim_train
def predict_rating(user_id, product_id):
    if user_id not in user_to_idx or product_id not in prod_to_idx:
        return np.nan
    uidx = user_to_idx[user_id]; pidx = prod_to_idx[product_id]
    user_ratings = R_train[uidx].toarray().flatten()
    rated_idx = np.where(user_ratings > 0)[0]
    if len(rated_idx) == 0:
        return np.nan
    sim_j = item_sim_train[pidx, rated_idx]
    numer = np.dot(sim_j, user_ratings[rated_idx])
    denom = np.sum(np.abs(sim_j))
    return numer/denom if denom != 0 else np.nan

# compute RMSE on a sample of test rows (to save time)
sample_test = test_df.sample(min(2000, len(test_df)), random_state=42)
y_true = []
y_pred = []
for _, row in sample_test.iterrows():
    pred = predict_rating(row['user_id'], row['product_id'])
    if np.isnan(pred):
        continue
    y_true.append(row['rating'])
    y_pred.append(pred)
print("Evaluated on", len(y_true), "rows")
print("RMSE:", np.sqrt(mean_squared_error(y_true, y_pred)))


Evaluated on 1623 rows
RMSE: 1.2194942087313156


In [ ]:
# Optional: save sample recommendations to csv
sample_user = user_ids[0]
recs = get_user_top_n_recs(sample_user, top_n=20)
pd.DataFrame({'user_id': sample_user, 'recommended_product': recs}).to_csv('sample_recs.csv', index=False)
print("Sample recommendations saved to sample_recs.csv")


Sample recommendations saved to sample_recs.csv
